# Mental Health Treatment Analysis — Machine Learning on Survey Data

**Mental health · Public health analytics · Classification · Survey data · Responsible ML**

This notebook explores a public mental health survey dataset to identify variables associated with whether respondents reported seeking mental health treatment.

The project was originally developed as a student project for the **Machine Learning II** module of **Santander Coders 2023 | Data Science**. This revised version is curated for a professional portfolio and emphasizes reproducibility, responsible interpretation, and methodological transparency.

> **Responsible interpretation note**  
> This analysis is not intended for diagnosis, clinical screening, or individual-level decision-making. The dataset is based on self-reported survey responses, and results should be interpreted as exploratory and dataset-specific. Model outputs should not be treated as causal explanations or clinical recommendations.


## 1. Project Objectives

### General objective

Develop and evaluate machine learning models to explore a classification problem related to reported mental health treatment-seeking behavior.

### Specific objectives

1. Load and inspect a public mental health survey dataset.
2. Conduct exploratory data analysis.
3. Define and justify the target variable.
4. Prepare data for supervised learning.
5. Train and evaluate a supervised classification model.
6. Address class imbalance within the modeling workflow.
7. Tune hyperparameters and compare model performance.
8. Inspect model behavior using feature importance with appropriate caution.
9. Explore unsupervised clustering methods as a methodological exercise.
10. Discuss limitations and responsible interpretation.


## 2. Dataset

The dataset is a public Kaggle survey dataset related to mental health, work context, demographic characteristics, and use of mental health services.

Typical variables include information such as gender, country, occupation, self-employment status, family history of mental health issues, perceived stress, changes in habits, coping struggles, and whether respondents reported seeking mental health treatment.

Because the dataset is based on self-reported survey responses, it should not be considered representative of broader populations without further validation.

**Data source:** https://www.kaggle.com/datasets/divaniazzahra/mental-health-dataset


## 3. Setup

The notebook is designed to run with a local copy of the dataset. Update `DATA_PATH` if your file has a different name or location.


In [ ]:
from pathlib import Path
import warnings
import re

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    precision_recall_fscore_support,
)
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError(
        "This notebook uses XGBoost. Install it with `pip install xgboost` before running."
    ) from exc

warnings.filterwarnings("ignore")

SEED = 42
pd.set_option("display.max_columns", 100)


## 4. Load Data

This section loads the dataset and standardizes column names to make downstream processing easier and more reproducible.


In [ ]:
DATA_PATH = Path("mental_health_dataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. "
        "Download the Kaggle dataset and update DATA_PATH accordingly."
    )

df_raw = pd.read_csv(DATA_PATH)

def clean_column_name(col: str) -> str:
    # Convert column names to a consistent snake_case format.
    col = col.strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col

df = df_raw.copy()
df.columns = [clean_column_name(c) for c in df.columns]

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
df.head()


## 5. Initial Data Inspection

The goal here is to understand structure, missingness, variable types, and the distribution of the target variable before modeling.


In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(dropna=True),
}).sort_values("missing_pct", ascending=False)

summary.head(20)


In [ ]:
TARGET = "treatment"

if TARGET not in df.columns:
    raise ValueError(f"Target column `{TARGET}` was not found. Available columns: {list(df.columns)}")

# Standardize target values and keep only valid binary responses.
df[TARGET] = df[TARGET].astype(str).str.strip().str.lower()
df_model = df[df[TARGET].isin(["yes", "no"])].copy()
df_model[TARGET] = df_model[TARGET].map({"yes": 1, "no": 0})

target_distribution = df_model[TARGET].value_counts(normalize=True).rename("proportion").to_frame()
target_distribution


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df_model, x=TARGET, ax=ax)
ax.set_title("Distribution of Target Variable: Reported Treatment-Seeking")
ax.set_xlabel("Reported treatment-seeking behavior (0 = No, 1 = Yes)")
ax.set_ylabel("Count")
plt.show()


## 6. Exploratory Data Analysis

This section examines selected variables and their relationship with the target. The goal is exploratory: to identify patterns worth considering during model interpretation, not to establish causality.


In [ ]:
variables_to_inspect = [
    "gender", "country", "occupation", "self_employed", "family_history",
    "days_indoors", "growing_stress", "changes_habits", "mental_health_history",
    "mood_swings", "coping_struggles", "work_interest", "social_weakness",
    "mental_health_interview", "care_options"
]

available_vars = [col for col in variables_to_inspect if col in df_model.columns]
available_vars


In [ ]:
def plot_target_share_by_category(data: pd.DataFrame, column: str, target: str = TARGET, max_categories: int = 12):
    # Plot treatment-seeking share by category for a selected categorical variable.
    temp = data[[column, target]].copy()
    temp[column] = temp[column].astype(str).fillna("missing")

    top_categories = temp[column].value_counts().head(max_categories).index
    temp = temp[temp[column].isin(top_categories)]

    rates = temp.groupby(column)[target].mean().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(8, 4))
    rates.plot(kind="bar", ax=ax)
    ax.set_title(f"Share Reporting Treatment by {column}")
    ax.set_ylabel("Share reporting treatment")
    ax.set_xlabel(column)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

for col in ["family_history", "care_options", "changes_habits", "coping_struggles"]:
    if col in available_vars:
        plot_target_share_by_category(df_model, col)


## 7. Data Preparation for Supervised Learning

The target variable is whether a respondent reported seeking mental health treatment. The model is trained as an exploratory classifier, not as a clinical decision tool.


In [ ]:
columns_to_drop = [TARGET]
columns_to_drop += [col for col in df_model.columns if "timestamp" in col or col in ["date", "time"]]

X = df_model.drop(columns=columns_to_drop, errors="ignore")
y = df_model[TARGET]

categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=SEED,
    stratify=y,
)

try:
    categorical_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    categorical_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", categorical_encoder),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)


## 8. Supervised Model: XGBoost Classifier

The supervised model is implemented as a pipeline that includes preprocessing, class balancing with SMOTE, and classification.

SMOTE is applied only inside the training workflow to avoid data leakage.


In [ ]:
xgb_model = XGBClassifier(
    random_state=SEED,
    eval_metric="logloss",
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
)

model_pipeline = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=SEED)),
    ("classifier", xgb_model),
])

model_pipeline.fit(X_train, y_train)

y_pred = model_pipeline.predict(X_test)
y_proba = model_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("AUC-ROC:", round(roc_auc_score(y_test, y_proba), 4))


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
ax.set_title("Confusion Matrix — XGBoost")
plt.show()

fpr, tpr, _ = roc_curve(y_test, y_proba)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_proba):.3f}")
ax.plot([0, 1], [0, 1], linestyle="--")
ax.set_title("ROC Curve — XGBoost")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()
plt.show()


## 9. Hyperparameter Tuning

A small grid search is used to demonstrate hyperparameter tuning. The parameter grid can be expanded in future work, but this portfolio version prioritizes readability and reproducibility.


In [ ]:
param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [3, 4],
    "classifier__learning_rate": [0.05, 0.1],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

grid_search = GridSearchCV(
    estimator=model_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=0,
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validated AUC-ROC:", round(grid_search.best_score_, 4))

best_model = grid_search.best_estimator_
best_proba = best_model.predict_proba(X_test)[:, 1]
best_pred = best_model.predict(X_test)

print(classification_report(y_test, best_pred))
print("Test AUC-ROC:", round(roc_auc_score(y_test, best_proba), 4))


## 10. Threshold Analysis

For sensitive domains such as mental health, evaluation should not rely only on default thresholds. This section compares precision, recall, and F1-score across candidate thresholds.

This remains an exploratory exercise and does not define a clinically valid decision threshold.


In [ ]:
thresholds = np.arange(0.1, 0.91, 0.05)
rows = []

for threshold in thresholds:
    pred_threshold = (best_proba >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        pred_threshold,
        average="binary",
        zero_division=0,
    )
    rows.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    })

threshold_results = pd.DataFrame(rows)
threshold_results.head()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(threshold_results["threshold"], threshold_results["precision"], marker="o", label="Precision")
ax.plot(threshold_results["threshold"], threshold_results["recall"], marker="o", label="Recall")
ax.plot(threshold_results["threshold"], threshold_results["f1"], marker="o", label="F1-score")
ax.set_title("Precision, Recall and F1-score by Classification Threshold")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.legend()
plt.show()


## 11. Feature Importance

Feature importance can help inspect model behavior, but it should be interpreted carefully. It does not establish causality and may be affected by preprocessing, encoding choices, class imbalance, and correlations between variables.


In [ ]:
def get_feature_names_from_preprocessor(fitted_preprocessor: ColumnTransformer) -> list:
    # Return transformed feature names from the fitted ColumnTransformer.
    feature_names = []

    if numeric_features:
        feature_names.extend(numeric_features)

    if categorical_features:
        cat_pipeline = fitted_preprocessor.named_transformers_["cat"]
        encoder = cat_pipeline.named_steps["encoder"]
        encoded_names = encoder.get_feature_names_out(categorical_features)
        feature_names.extend(encoded_names.tolist())

    return feature_names

fitted_preprocessor = best_model.named_steps["preprocessor"]
fitted_classifier = best_model.named_steps["classifier"]

feature_names = get_feature_names_from_preprocessor(fitted_preprocessor)
importances = fitted_classifier.feature_importances_

importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(data=importance_df, x="importance", y="feature", ax=ax)
ax.set_title("Top Feature Importances — Exploratory Interpretation")
ax.set_xlabel("Importance")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()


## 12. Unsupervised Learning: Clustering Exploration

Unsupervised learning was explored as a methodological exercise. The purpose was to examine whether the available variables formed clear groups after preprocessing.

The clustering outputs should not be interpreted as clinically meaningful groups without further validation.


In [ ]:
X_processed = preprocessor.fit_transform(X)

pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_processed)

cluster_results = []

for k in range(2, 8):
    kmeans = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = kmeans.fit_predict(X_processed)
    cluster_results.append({
        "model": "KMeans",
        "parameter": f"k={k}",
        "silhouette": silhouette_score(X_processed, labels),
        "davies_bouldin": davies_bouldin_score(X_processed, labels),
    })

cluster_results_df = pd.DataFrame(cluster_results)
cluster_results_df


In [ ]:
best_k = cluster_results_df.sort_values("silhouette", ascending=False).iloc[0]["parameter"]
best_k = int(best_k.split("=")[1])

kmeans = KMeans(n_clusters=best_k, random_state=SEED, n_init=10)
kmeans_labels = kmeans.fit_predict(X_processed)

fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=kmeans_labels, ax=ax, legend=False)
ax.set_title(f"K-Means Clusters Visualized with PCA (k={best_k})")
ax.set_xlabel("PCA 1")
ax.set_ylabel("PCA 2")
plt.show()


In [ ]:
# DBSCAN is sensitive to parameter choices and high-dimensional encoded data.
# This example is exploratory only.
dbscan = DBSCAN(eps=0.8, min_samples=10)
dbscan_labels = dbscan.fit_predict(X_processed)

n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = np.sum(dbscan_labels == -1)

print("Estimated clusters:", n_clusters)
print("Noise points:", n_noise)

fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=dbscan_labels, ax=ax, legend=False)
ax.set_title("DBSCAN Clusters Visualized with PCA — Exploratory")
ax.set_xlabel("PCA 1")
ax.set_ylabel("PCA 2")
plt.show()


## 13. Key Findings

The supervised model identified statistical patterns associated with reported treatment-seeking behavior in the dataset. In exploratory feature-importance analysis, variables related to family history, work context, and self-reported changes in habits or mental health-related experiences may contribute to model predictions.

However, these findings are **dataset-specific** and should be interpreted with caution. They do not establish causality, clinical relevance, or generalizable population-level conclusions.

The clustering analysis was useful as a methodological exercise, but cluster separation should be validated carefully before any substantive interpretation.


## 14. Limitations

This project has important limitations:

- The dataset is based on self-reported survey responses.
- The data may not be representative of broader populations.
- The models identify statistical patterns but do not establish causal relationships.
- Feature importance should not be interpreted as clinical explanation.
- Model performance may be affected by survey design, missing values, class imbalance, and encoding choices.
- The analysis should not be used for diagnosis, individual-level decision-making, or clinical screening.
- Further validation and ethical review would be required before any real-world use in health contexts.


## 15. Responsible Data and AI Interpretation

Because this project deals with mental health-related data, the analysis follows a cautious interpretation framework:

- Avoid interpreting predictions as diagnoses.
- Avoid treating model outputs as clinical recommendations.
- Distinguish association from causality.
- Recognize possible bias in self-reported survey data.
- Interpret feature importance carefully.
- Communicate findings as exploratory and dataset-specific.
- Consider the ethical implications of applying machine learning to mental health data.


## 16. Next Steps

Potential improvements include:

- Compare additional supervised models under a consistent validation strategy.
- Explore model explainability methods, such as SHAP, with careful interpretation.
- Test model stability across multiple random seeds and train-test splits.
- Improve documentation of preprocessing and feature-selection decisions.
- Investigate whether clustering methods add meaningful analytical value after more targeted feature selection.
- Review the dataset metadata and sampling process in greater depth before drawing policy-oriented conclusions.


## 17. Reproducibility Notes

Suggested minimal dependencies:

```text
pandas
numpy
matplotlib
seaborn
scikit-learn
imbalanced-learn
xgboost
```

This notebook does not include the dataset. Download it from the public Kaggle source and update `DATA_PATH` before running.
